# 03: Lakeflow Connect

**Exam objectives (from Data Ingestion and Loading domain):**
- Configure Lakeflow Connect to reliably ingest data from diverse   enterprise sources into Unity-Catalog-governed tables.
- Prioritize between Auto Loader, Lakeflow Connect (standard and managed connectors), partner connectors, and other ingestion methods based on technical requirements such as data volume, ingestion frequency, data types, and governance needs.

**Free Edition note:** Managed connectors require external SaaS or database sources not available on Free Edition. This notebook is conceptual — covering connector types, decision frameworks, and terminology — without hands-on runs.

## Standard vs. Managed Connectors

Lakeflow Connect has two connector families, distinguished by where the source data lives.

### Standard connectors

**Source type:** Cloud object storage holding files in open formats.

**Examples:** S3 directories of JSON, ADLS containers of Parquet, GCS buckets of CSV.

**Mechanism:** Built on the same primitives covered in exercises 01 and 02 — Auto Loader for incremental file ingestion, COPY INTO for SQL-based incremental ingestion. Lakeflow Connect packages these with configuration, orchestration, and governance integration.

**When to use:** When the source data is already files in cloud storage. This is the case for raw data dumps, exports from upstream systems, log files, sensor data, and anything else landed as files.

**Setup:** Configure source path, file format, target table, schedule.

### Managed connectors

**Source type:** External SaaS applications and operational databases with their own APIs.

**Examples:** Salesforce, Workday, ServiceNow, SQL Server, SharePoint, Google Analytics, NetSuite, MySQL.

**Mechanism:** Databricks handles end-to-end integration — connection management, authentication, schema discovery, change-data-capture, pagination, rate limiting, scheduling. The connector knows the source system's semantics and uses the appropriate API patterns to extract data efficiently.

**When to use:** When the source is a system with an API rather than a file path. Especially appropriate for sources requiring CDC (change data capture), complex authentication, or schema discovery.

**Setup:** Configure source credentials, select objects/tables to ingest, target catalog/schema, schedule. The connector does the rest.

### The decision rule

Source is files in cloud storage → standard connector.

Source is a SaaS app or operational database → managed connector.

If a managed connector exists for your source, prefer it over a partner connector (third-party ETL tool). Managed connectors integrate directly with UC, run inside Databricks compute, and don't require managing a separate vendor relationship.

## Why managed connectors matter

Before Lakeflow Connect's managed connectors, the standard pattern for ingesting from SaaS sources was:

1. Use a third-party ETL tool (Fivetran, Airbyte, Stitch) to extract from the SaaS source and land files in cloud storage.
2. Use Auto Loader or COPY INTO to ingest those files into Delta tables.
3. Apply UC governance separately as a downstream step.

This approach has tradeoffs:
- Two systems to manage and pay for (the ETL tool + Databricks).
- Data sits in cloud storage between the two systems, complicating governance and lineage.
- The ETL tool's authentication, schema, and CDC behavior is opaque to Databricks.

Managed connectors collapse this into one system. The connector runs inside Databricks compute, lands data directly in UC-governed tables, and exposes its operational state (schema changes, sync history, errors) through standard Databricks tooling.

For the exam: when given a scenario involving a SaaS source where both partner and managed connector options exist, the modern answer is the managed connector.

## Choosing an Ingestion Method

Five options to choose from, each with a sweet spot:

| Method | Source type | Interface | Latency | Best for |
|--------|------------|-----------|---------|----------|
| Auto Loader | Files in cloud storage | PySpark | Streaming or batch | High-volume file ingestion, schema evolution, streaming or near-real-time |
| COPY INTO | Files in cloud storage | SQL | Batch only | SQL-first teams, scheduled batch loads, small-to-medium file volumes |
| Lakeflow Connect (standard) | Files in cloud storage | Config + UI/API | Streaming or batch | Same as Auto Loader/COPY INTO but with managed orchestration and governance |
| Lakeflow Connect (managed) | SaaS APIs and operational databases | Config + UI/API | Scheduled CDC | Salesforce, Workday, ServiceNow, SQL Server, etc. |
| Partner connectors | SaaS APIs (third-party tools) | External (Fivetran, etc.) | Varies | Legacy integrations; sources no managed connector covers |

## Decision framework

Answer in order. The first match wins.

**1. Is the source a SaaS application or operational database (not files)?**
   - Yes → Lakeflow Connect managed connector (if one exists for the source). Otherwise partner connector.
   - No → continue.

**2. Is the source files in cloud storage?**
   - Yes → continue to question 3.

**3. Does the team prefer SQL or PySpark?**
   - SQL → COPY INTO (for batch) or Lakeflow Connect standard with SQL config (for orchestrated batch).
   - PySpark → Auto Loader.
   - Either → consider latency requirement next.

**4. What's the latency requirement?**
   - Batch (hourly or slower) → COPY INTO or Auto Loader with trigger(availableNow=True).
   - Near-real-time (minutes) → Auto Loader with continuous trigger.
   - Real-time (sub-second) → Structured Streaming with Kafka or equivalent (outside Auto Loader's scope).

**5. What's the file volume?**
   - Small to medium (< 1M files in directory) → either Auto Loader directory listing or COPY INTO.
   - Large (millions+) → Auto Loader with file notification mode specifically — directory listing scales poorly.

**6. Does the pipeline need complex transformation logic before write?**
   - Yes → Auto Loader feeding a Structured Streaming query (full DataFrame API available).
   - No → COPY INTO or Auto Loader with simple write are both fine.

## Scenarios

**Scenario A:** A SaaS marketing team wants to ingest customer interaction data from Salesforce into Databricks for downstream BI dashboards. They need updates every 4 hours.
→ Lakeflow Connect managed connector (Salesforce). Question 1 hits.

**Scenario B:** A nightly batch job ingests roughly 500 JSON files per night from an S3 prefix. The team writes SQL and wants minimal orchestration overhead.
→ COPY INTO. Question 1 no, question 2 yes, question 3 SQL preference, 
question 4 batch.

**Scenario C:** A pipeline ingests files from S3 into Delta tables. The S3 prefix contains 50 million files and grows by 10K/day. The pipeline needs to deduplicate and join with a lookup table before writing.
→ Auto Loader with file notification mode, feeding a Structured Streaming query that does the dedup and join. Question 1 no, question 2 yes, question 5 says file notification, question 6 says full DataFrame API.

**Scenario D:** A team needs to ingest Workday HR data daily. A managed connector for Workday exists.
→ Lakeflow Connect managed connector. Question 1 hits.

**Scenario E:** A team needs to ingest data from a proprietary internal system with a custom REST API. No managed or partner connector exists.
→ Custom PySpark code calling the API, landing results in cloud storage as files, then Auto Loader picks them up. This is the "escape hatch" pattern — managed/partner connectors are preferred when they exist, custom code when they don't.

## Things the exam likes to test that trip people up

1. **"Auto Loader is for files; Lakeflow Connect is for everything else"** is wrong. Lakeflow Connect standard connectors *also* handle files, with Auto Loader as the underlying engine. The distinction is whether you want to write Auto Loader code yourself (Auto Loader directly) or configure a pipeline that orchestrates it (Lakeflow Connect standard).

2. **"Use partner connectors for SaaS sources"** is outdated advice. Managed connectors are now the recommended choice when one exists for the source. The exam tests this because the recommendation recently flipped.

3. **"COPY INTO is the same as Auto Loader's batch mode"** is mostly right but not exactly. Both are batch ingestion, both are idempotent, both handle incremental files. The differences are: interface (SQL vs. PySpark), state location (table transaction log vs. checkpoint), and Auto Loader's broader feature set (file notification, schema 
evolution modes). The exam may ask which to choose, and "SQL vs. PySpark preference" plus "expected scale" are the deciding factors.

4. **Lakeflow Connect is a service, not a syntax.** There's no SQL keyword `LAKEFLOW CONNECT`. It's an umbrella product configured through UI, API, or asset bundles. The connectors *underneath* may use Auto Loader or other primitives, but Lakeflow Connect itself is a configuration-and-orchestration layer.

## Self Check Questions

1. Lakeflow Connect has two connector families — standard and managed. What's the fundamental distinction between them? Map each of these sources to the right connector family: a directory of Parquet files in S3, a Salesforce instance, a SQL Server database, an ADLS container of CSV files, a Workday HR system.

2. A team is currently using Fivetran to pull data from Salesforce into   S3, then using Auto Loader to ingest those files into Delta tables. They're reviewing whether to migrate to Lakeflow Connect's managed Salesforce connector. What are the practical advantages of the migration? Is there a scenario where they should stay on Fivetran?

3. You've covered four file-based ingestion options in this domain: Auto Loader, COPY INTO, Lakeflow Connect standard, and partner connectors. Two are SQL-based, two are PySpark-based — categorize each. Then: a team that strongly prefers SQL is ingesting JSON files from S3 on a nightly schedule. Which would you recommend, and what would push you toward the other SQL option instead?

4. A team needs to ingest data from a custom internal REST API. No managed connector exists for this system, no partner connector covers it either. What's the right ingestion pattern? Reason about this — there's no single right answer, but there's a recognizable pattern.

5. The exam objective uses the phrase "diverse enterprise sources." Why is this language specifically associated with managed connectors and not with Auto Loader or COPY INTO? What's the underlying capability difference?

6. A pipeline ingests 50,000 small JSON files per day from an S3 directory that already contains 200 million historical files. The pipeline runs hourly. Which ingestion method, and which specific configuration of that method, would you recommend?

## Self Check Answers

1. Lakeflow Connect standard connectors are suited for file ingestions (CSV, Parquet, JSON, etc.) from a cloud storage layer. Lakeflow Connect managed connectors are suited for ingesting data in a non-file format from SaaS applications or operational databases, if Databricks offers an official connector for the application. 
    1. Directory of Parquet files -> Standard
    2. A Salesforce instance -> Managed
    3. A SQL Server database -> Managed 
    4. An ADLS container of CSV files-> Standard
    5. A Workday HR system -> Managed

2. The practical advantage of using the Saleforce managed connector is twofold: 
    1. Using a managed connector allows UC operational state tracking and governance throughout the ingestion process, and allows the data to be ingested directly into UC governed tables. This dissolves the need for the second load to Delta.
    2. The team no longer needs to pay for Fivetran if all relevant ingestions can be substitued with a managed connector.
    3. However, sticking with Fivetran may be the better option if Legacy integrations are necessary where there is not an available managed connector. Another consideration is source coverage, as Fivetran's connectivity extends to hundreds of sources whereas Lakeflow Connect's managed connector catalog is much smaller (but growing).

3. Auto Loader is PySpark based, COPY INTO and Lakeflow Connect standard connectors (with SQL config) are SQL based, and Partner Connectors are neither. A SQL-centric team with a nightly ingestion from S3 should used the COPY INTO for a nightly batch job. A Lakeflow Connect standard connector configured for S3 JSON files would be the suitable option if the expected latency was near-real-time or real-time, as Lakeflow Connect offers streaming capabilities. Standard connectors also offers:
    - Managed orchestration instead of maintaining the COPY INTO job in their own scheduler
    - UC governance integration from configuration rather than configuring it separately
    - Anticipated growth in pipeline complexity is better handled by Lakeflow Connect's feature set

4. One acceptable pattern is to utilize PySpark to call the API, land the data in cloud storage as files, and lastly use Auto Loader to ingest the files into tables.
    - One thing worth adding for completeness (not a correction): this approach effectively decomposes the "ingest from custom source" problem into two problems that Databricks already solves well — (a) "get data into cloud storage" (custom PySpark) and (b) "ingest from cloud storage" (Auto Loader). It's a layering pattern, and the value is that you keep Databricks-native ingestion for the half it does well rather than rolling everything yourself.

5. Auto Loader and COPY INTO are only capable of ingesting data from cloud storage layers whereas manager connectors enable connections to external applications and operational databases. Lakeflow Connect can leverage Auto Loader during ingestion, but Auto Loader on its own does not have the scope to load data from an external application.

6. I would recommend Auto Loader with the file notification configuration. Directly Listing at this scale is nonsensical and Auto Loader only needs to be aware of the new files, not 200 million + old files.
    - One thing worth knowing for completeness: at 200M files, you'd also typically want to think about checkpoint location stability (very important to protect) and possibly about partitioning the source path to limit what Auto Loader has to track. But for the question as asked, "Auto Loader with file notification" is the full answer.